# Phenomenological ESR frequency sweep

This example simulates the ESR spectrum of a spin-1/2 using thermal
populations, transition matrix elements, and a prescribed Lorentzian
linewidth. The signal is phenomenological and does not represent an absolute current. The notebook contains:
1. Initializing a Spin-1/2 with Zeeman energies
2. Simulating the phenomenological ESR spectrum
3. Plot the dependence of the ESR-spectrum on the tip-magnetic field

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import ESR, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Initialize a single spin-1/2

Adding the Zeeman energy to the Hamiltonian. A tilted field from the magnetic tip used in ESR-STM provides both
transverse driving and longitudinal detection contributions.

In [ ]:
spin = SpinSys([0.5])

# Hamiltonian parameters
spin.B = [0.0, 0.0, 0.5]  # T
spin.gfactorvector = [2.0]
spin.tip = "f"
spin.T = 0.5  # K

# Tip field: angle measured from z towards x
angle = 45.0  # degrees
B_tip = 0.02  # T

spin.BTip = [
    B_tip * np.sin(np.deg2rad(angle)),
    0.0,
    B_tip * np.cos(np.deg2rad(angle)),
]

# Initial calculations
hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)
bath.calc_Populations(spin, AllowPumping=False)

f0_theory = abs(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])

print(f"Resonance frequency: {f0_theory:.6f} GHz")
print(f"Thermal populations: {spin.Populations}")

## Frequency sweep

Each transition contributes a Lorentzian with FWHM `lw`. Its weight
combines the population difference, driving matrix element, and detection
contrast. The linewidth is an input, not calculated from relaxation times.

In [ ]:
freq_array = np.linspace(12.0, 16.0, 2000)  # GHz
lw = 0.05  # GHz; FWHM

# The plot function returns a dictionary with the figure and data 
return_dict = ESR.plot_FreqSweep(
    spin,
    freq_array=freq_array,
    AllowPumping=False, #  If this is set to True, the populations will be computed from the tunneling parameters.
    lw=lw,
    norm=True,
)

## Tip-field map

Sweep the tip-field strength at a fixed angle. The tip field changes both
the resonance frequency and transition weight. Using `norm=False`
preserves relative intensities between field values.

In [ ]:
BTip_array = np.linspace(0.005, 0.08, 201)  # T

return_dict = ESR.plot_Tipfieldmap(
    spin,
    freq_array=freq_array,
    BTip_array=BTip_array,
    lw=lw,
    norm=False,
    AllowPumping=False,
    Angle=angle,
)